In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D12 — Our World in Data Annual CO2 Emissions Dataset
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter
import hashlib
import json
import re
import unicodedata

import pandas as pd

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

DOCUMENT_ID = "D12"
DOCUMENT_NAME = "Our World in Data — Annual CO2 emissions time series"

BRANCH = "A"
BRANCH_NAME = "Direct Ingestion"
INPUT_REPRESENTATION = "Original CSV file"

EXPECTED_SOURCE_SHA256 = (
    "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788"
)

EXPECTED_RECORD_COUNT = 25

REFERENCE_CATEGORY = "Environmental time-series"
REFERENCE_TOPIC = "Annual CO2 emissions"
REFERENCE_DESCRIPTION = "Annual CO2 emissions"
REFERENCE_UNIT = None

TARGET_YEARS = [
    1750, 1800, 1850, 1900, 1950,
    1960, 1970, 1980, 1990, 2000,
    2010, 2011, 2012, 2013, 2014,
    2015, 2016, 2017, 2018, 2019,
    2020, 2021, 2022, 2023, 2024
]

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

ALIGNMENT_IDENTITY_FIELDS = [
    "Reporting Period"
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Source Location"
]

EXPECTED_CATEGORY_COUNTS = {
    REFERENCE_CATEGORY: EXPECTED_RECORD_COUNT
}

OUTPUT_DIR = Path("outputs_D12_validation_A_revised")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "detailed":
        OUTPUT_DIR / "D12_branch_A_validation_detailed.csv",
    "discrepant":
        OUTPUT_DIR / "D12_branch_A_discrepant_records.csv",
    "missing":
        OUTPUT_DIR / "D12_branch_A_missing_records.csv",
    "unsupported":
        OUTPUT_DIR / "D12_branch_A_unsupported_records.csv",
    "alignment_issues":
        OUTPUT_DIR / "D12_branch_A_alignment_issues.json",
    "reference_semantics":
        OUTPUT_DIR / "D12_reference_semantics_confirmation.json",
    "summary":
        OUTPUT_DIR / "D12_branch_A_validation_summary.json"
}

print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 3. Upload the four canonical validation inputs
# ============================================================

print(
    "Upload exactly these four files:\n"
    "1. D12_reference_values.csv\n"
    "2. D12_branch_A_parsed_extraction.json\n"
    "3. D12_branch_A_technical_diagnostics.json\n"
    "4. D12_branch_A_experiment_metadata.json"
)

uploaded = files.upload()

def require_file(expected_name):
    matches = [
        Path(name)
        for name in uploaded
        if Path(name).name == expected_name
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one file named {expected_name}; "
            f"found {len(matches)}."
        )
    return matches[0]

REFERENCE_PATH = require_file("D12_reference_values.csv")
EXTRACTION_PATH = require_file("D12_branch_A_parsed_extraction.json")
TECHNICAL_DIAGNOSTICS_PATH = require_file("D12_branch_A_technical_diagnostics.json")
METADATA_PATH = require_file("D12_branch_A_experiment_metadata.json")

print("Canonical inputs resolved.")

In [ ]:
# ============================================================
# 4. Load and fingerprint inputs
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

REFERENCE_SHA256 = sha256_file(REFERENCE_PATH)
EXTRACTION_SHA256 = sha256_file(EXTRACTION_PATH)
TECHNICAL_DIAGNOSTICS_SHA256 = sha256_file(TECHNICAL_DIAGNOSTICS_PATH)
METADATA_SHA256 = sha256_file(METADATA_PATH)

reference_df = pd.read_csv(
    REFERENCE_PATH,
    encoding="utf-8-sig",
    keep_default_na=False
)

with EXTRACTION_PATH.open("r", encoding="utf-8") as f:
    extraction_payload = json.load(f)

with TECHNICAL_DIAGNOSTICS_PATH.open("r", encoding="utf-8") as f:
    technical_diagnostics = json.load(f)

with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata_payload = json.load(f)

extracted_records = extraction_payload.get("records", [])
extracted_df = pd.DataFrame(extracted_records)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Parsed extraction SHA-256:", EXTRACTION_SHA256)
print("Structure SHA-256:", TECHNICAL_DIAGNOSTICS_SHA256)
print("Metadata SHA-256:", METADATA_SHA256)

In [ ]:
# ============================================================
# 5. Confirm fixed Stage 1 reference semantics
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist() == EXPECTED_FIELDS
)

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"].value_counts().to_dict()
    if "Category" in reference_df.columns
    else {}
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

reference_topic_constant = bool(
    (reference_df["Topic"] == REFERENCE_TOPIC).all()
)

reference_description_constant = bool(
    (reference_df["Description"] == REFERENCE_DESCRIPTION).all()
)

# CSV loading with keep_default_na=False turns null cells into "".
reference_unit_null = bool(
    reference_df["Unit"].apply(
        lambda value: value is None
        or (isinstance(value, str) and value.strip() == "")
        or pd.isna(value)
    ).all()
)

reference_values_numeric = pd.to_numeric(
    reference_df["Value"],
    errors="coerce"
).notna().all()

reference_values_non_negative = bool(
    (
        pd.to_numeric(reference_df["Value"], errors="coerce")
        >= 0
    ).all()
)

reference_periods = pd.to_numeric(
    reference_df["Reporting Period"],
    errors="coerce"
)

reference_years_valid = (
    reference_periods.notna().all()
    and reference_periods.astype(int).tolist() == TARGET_YEARS
)

reference_identity_unique = (
    reference_df["Reporting Period"].astype(str).nunique()
    == EXPECTED_RECORD_COUNT
)

reference_source_locations_valid = bool(
    reference_df["Source Location"]
    .astype(str)
    .str.fullmatch(r"CSV data row \d+")
    .all()
)

reference_semantic_checks = {
    "reference_schema_exact":
        bool(reference_schema_exact),
    "reference_record_count_valid":
        bool(reference_record_count_valid),
    "reference_category_counts_valid":
        bool(reference_category_counts_valid),
    "reference_topic_constant":
        bool(reference_topic_constant),
    "reference_description_constant":
        bool(reference_description_constant),
    "reference_unit_null":
        bool(reference_unit_null),
    "reference_values_numeric":
        bool(reference_values_numeric),
    "reference_values_non_negative":
        bool(reference_values_non_negative),
    "reference_years_valid":
        bool(reference_years_valid),
    "reference_identity_unique":
        bool(reference_identity_unique),
    "reference_source_locations_valid":
        bool(reference_source_locations_valid)
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

REFERENCE_SEMANTICS_CONFIRMATION = {
    "document_id": DOCUMENT_ID,
    "reference_semantics_valid":
        bool(reference_semantics_valid),
    "checks":
        reference_semantic_checks
}

PATHS["reference_semantics"].write_text(
    json.dumps(
        REFERENCE_SEMANTICS_CONFIRMATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    REFERENCE_SEMANTICS_CONFIRMATION,
    ensure_ascii=False,
    indent=2
))

if not reference_semantics_valid:
    raise AssertionError(
        "D12 Stage 1 reference semantics are not valid."
    )

In [ ]:
# ============================================================
# 6. Confirm Branch A provenance and schema
# ============================================================

top_level_object_valid = isinstance(
    extraction_payload,
    dict
)

document_id_correct = (
    extraction_payload.get("document_id") == DOCUMENT_ID
)

branch_correct = (
    extraction_payload.get("branch") == BRANCH
)

records_is_list = isinstance(
    extraction_payload.get("records"),
    list
)

record_schema_valid = True
field_types_valid = True
schema_issue_rows = []

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

if records_is_list:
    for i, record in enumerate(extracted_records):
        if not isinstance(record, dict):
            record_schema_valid = False
            schema_issue_rows.append({
                "record_index": i,
                "issue": "record_not_object"
            })
            continue

        if list(record.keys()) != EXPECTED_FIELDS:
            record_schema_valid = False
            schema_issue_rows.append({
                "record_index": i,
                "issue": "field_names_or_order",
                "observed_fields": list(record.keys())
            })

        for field in STRING_OR_NULL_FIELDS:
            value = record.get(field)
            if value is not None and not isinstance(value, str):
                field_types_valid = False
                schema_issue_rows.append({
                    "record_index": i,
                    "issue": "field_type",
                    "field": field,
                    "observed_type": type(value).__name__
                })

        value = record.get("Value")
        if (
            value is not None
            and (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
            )
        ):
            field_types_valid = False
            schema_issue_rows.append({
                "record_index": i,
                "issue": "field_type",
                "field": "Value",
                "observed_type": type(value).__name__
            })
else:
    record_schema_valid = False
    field_types_valid = False

schema_validity = bool(
    branch_A_structurally_evaluable
)

metadata_parsed_hash = metadata_payload.get(
    "parsed_extraction_sha256"
)

metadata_source_hash = metadata_payload.get(
    "source_sha256"
)

branch_A_structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

parsed_extraction_hash_matches_metadata = (
    metadata_parsed_hash == EXTRACTION_SHA256
)

source_hash_matches_stage_1 = (
    metadata_source_hash == EXPECTED_SOURCE_SHA256
)

input_provenance = {
    "reference_file":
        REFERENCE_PATH.name,
    "reference_sha256":
        REFERENCE_SHA256,
    "parsed_extraction_file":
        EXTRACTION_PATH.name,
    "parsed_extraction_sha256":
        EXTRACTION_SHA256,
    "experiment_metadata_file":
        METADATA_PATH.name,
    "experiment_metadata_sha256":
        METADATA_SHA256,
    "branch_A_structurally_evaluable":
        branch_A_structurally_evaluable,
    "parsed_extraction_hash_matches_metadata":
        parsed_extraction_hash_matches_metadata,
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "technical_diagnostics_sha256":
        TECHNICAL_DIAGNOSTICS_SHA256,

    "branch_A_structurally_evaluable":
        branch_A_structurally_evaluable,
    "source_hash_matches_stage_1":
        source_hash_matches_stage_1
}

print("Schema validity:", schema_validity)
print(json.dumps(input_provenance, indent=2))

In [ ]:
# ============================================================
# 7. Comparison-only canonicalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    text = (
        text.replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("’", "'")
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def canonical_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    if re.fullmatch(r"\d{4}", text):
        return int(text)

    return text


def parse_numeric(value):
    if value is None or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip().replace(",", "")

    if re.fullmatch(r"[-+]?\d+(?:\.\d+)?", text):
        return float(text)

    return None


def canonical_null(value):
    if value is None:
        return None

    if isinstance(value, float) and pd.isna(value):
        return None

    if isinstance(value, str) and value.strip() == "":
        return None

    return normalise_text(value)


def canonical_source_location(value):
    text = normalise_text(value)

    if text is None:
        return None

    match = re.fullmatch(
        r"CSV data row (\d+)",
        text
    )

    return (
        f"CSV data row {int(match.group(1))}"
        if match
        else text
    )

In [ ]:
# ============================================================
# 8. Deterministic one-to-one alignment by Reporting Period
# ============================================================

def build_identity_index(records, dataset_name):
    index = {}
    duplicate_groups = []

    for record_index, record in enumerate(records):
        if not isinstance(record, dict):
            continue

        identity = canonical_period(
            record.get("Reporting Period")
        )

        if identity in index:
            duplicate_groups.append({
                "dataset": dataset_name,
                "identity": identity,
                "record_index": record_index
            })
        else:
            index[identity] = {
                "record_index": record_index,
                "record": record
            }

    return index, duplicate_groups


reference_records = reference_df.to_dict("records")

reference_index, reference_duplicates = (
    build_identity_index(
        reference_records,
        "Reference"
    )
)

extraction_index, extraction_duplicates = (
    build_identity_index(
        extracted_records,
        "Extraction"
    )
)

alignment_issues = (
    reference_duplicates
    + extraction_duplicates
)

PATHS["alignment_issues"].write_text(
    json.dumps(
        alignment_issues,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print("Alignment issues:", len(alignment_issues))

if reference_duplicates:
    raise AssertionError(
        "Reference identity is not unique."
    )

In [ ]:
# ============================================================
# 9. Match records and compare fields
# ============================================================

def compare_field(field, reference_value, extracted_value):
    if field == "Value":
        return (
            parse_numeric(reference_value)
            == parse_numeric(extracted_value)
        )

    if field == "Unit":
        return (
            canonical_null(reference_value)
            == canonical_null(extracted_value)
        )

    if field == "Reporting Period":
        return (
            canonical_period(reference_value)
            == canonical_period(extracted_value)
        )

    if field == "Source Location":
        return (
            canonical_source_location(reference_value)
            == canonical_source_location(extracted_value)
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


detailed_rows = []
discrepant_rows = []
missing_rows = []
unsupported_rows = []

all_identities = sorted(
    set(reference_index.keys())
    | set(extraction_index.keys()),
    key=lambda x: (x is None, str(x))
)

for identity in all_identities:
    ref_entry = reference_index.get(identity)
    ext_entry = extraction_index.get(identity)

    if ref_entry is not None and ext_entry is None:
        row = ref_entry["record"].copy()
        row["Reference Record Index"] = ref_entry["record_index"]
        missing_rows.append(row)
        continue

    if ext_entry is not None and ref_entry is None:
        row = ext_entry["record"].copy()
        row["Extracted Record Index"] = ext_entry["record_index"]
        unsupported_rows.append(row)
        continue

    ref_record = ref_entry["record"]
    ext_record = ext_entry["record"]

    field_results = {
        field: compare_field(
            field,
            ref_record.get(field),
            ext_record.get(field)
        )
        for field in EXPECTED_FIELDS
    }

    primary_correct = all(
        field_results[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    fully_correct = primary_correct

    identity_fields_match = all(
        field_results[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    detail = {
        "Reference Record Index":
            ref_entry["record_index"],
        "Extracted Record Index":
            ext_entry["record_index"],
        "Identity Reporting Period":
            identity,
        "Fully Correct":
            fully_correct,
        "Primary Correct":
            primary_correct
    }

    for field in EXPECTED_FIELDS:
        detail[f"Reference {field}"] = ref_record.get(field)
        detail[f"Extracted {field}"] = ext_record.get(field)
        detail[f"{field} Correct"] = field_results[field]

    detailed_rows.append(detail)

    if not fully_correct:
        discrepant_rows.append(detail.copy())

detailed_df = pd.DataFrame(detailed_rows)
discrepant_df = pd.DataFrame(discrepant_rows)
missing_df = pd.DataFrame(missing_rows)
unsupported_df = pd.DataFrame(unsupported_rows)

print("Aligned:", len(detailed_df))
print("Discrepant:", len(discrepant_df))
print("Missing:", len(missing_df))
print("Unsupported/unmatched:", len(unsupported_df))

In [ ]:
# ============================================================
# 10. Calculate common validation metrics
# ============================================================

reference_count = len(reference_df)
extracted_count = len(extracted_records)
aligned_count = len(detailed_df)

fully_correct_count = (
    int(detailed_df["Fully Correct"].sum())
    if aligned_count
    else 0
)

discrepant_count = len(discrepant_df)
missing_count = len(missing_df)
unsupported_count = len(unsupported_df)

completeness = (
    aligned_count / reference_count
    if reference_count
    else None
)

record_precision_exact = (
    fully_correct_count / extracted_count
    if extracted_count
    else 0.0
)

record_recall_exact = (
    fully_correct_count / reference_count
    if reference_count
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)

primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:
    if aligned_count:
        primary_field_accuracy[field] = float(
            detailed_df[
                f"{field} Correct"
            ].mean()
        )
    else:
        primary_field_accuracy[field] = None

valid_primary_values = [
    value
    for value in primary_field_accuracy.values()
    if value is not None
]

field_accuracy = (
    sum(valid_primary_values)
    / len(valid_primary_values)
    if valid_primary_values
    else None
)

category_metrics = {}

for category in sorted(
    set(reference_df["Category"].astype(str))
):
    ref_category_count = int(
        (reference_df["Category"] == category).sum()
    )

    ext_category_count = sum(
        1
        for record in extracted_records
        if isinstance(record, dict)
        and record.get("Category") == category
    )

    aligned_category = detailed_df[
        detailed_df["Reference Category"] == category
    ] if aligned_count else pd.DataFrame()

    aligned_category_count = len(aligned_category)

    fully_correct_category = (
        int(aligned_category["Fully Correct"].sum())
        if aligned_category_count
        else 0
    )

    p = (
        fully_correct_category / ext_category_count
        if ext_category_count
        else 0.0
    )
    r = (
        fully_correct_category / ref_category_count
        if ref_category_count
        else 0.0
    )
    f1 = (
        2 * p * r / (p + r)
        if p + r
        else 0.0
    )

    category_metrics[category] = {
        "expected_records": ref_category_count,
        "extracted_records": ext_category_count,
        "aligned_records": aligned_category_count,
        "fully_correct_records": fully_correct_category,
        "discrepant_records":
            aligned_category_count - fully_correct_category,
        "completeness":
            aligned_category_count / ref_category_count
            if ref_category_count
            else None,
        "record_precision_exact": p,
        "record_recall_exact": r,
        "record_f1_exact": f1
    }

In [ ]:
# ============================================================
# 11. Create final validation summary
# ============================================================

comparison_rules = {
    "raw_extraction_modified":
        False,
    "manual_correction_applied":
        False,
    "comparison_normalisation_scope":
        "Comparison copies only",
    "identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,
    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,
    "value":
        (
            "Exact represented numeric equality after deterministic "
            "parsing; no tolerance, rescaling, interpolation, rounding "
            "or conversion."
        ),
    "unit":
        (
            "Exact null agreement. No unit is inferred because the "
            "source CSV contains no explicit measurement-unit field."
        ),
    "text_fields":
        (
            "Conservative normalised exact agreement for fixed "
            "Category, Topic and Description values."
        ),
    "source_location":
        "Exact physical CSV data-row agreement.",
    "d12_equivalence_rules_status":
        (
            "Frozen D12 document/schema-level comparison rules. "
            "No additional Branch-A-derived semantic equivalence rules "
            "were required. The fixed Stage 1 reference and these "
            "comparison rules must be reused unchanged for Branches B and C."
        ),
    "equivalence_rules_frozen":
        True
}

matching_rules = {
    "identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,
    "one_to_one_assignment":
        (
            "Unique deterministic Reporting Period (year) identity."
        ),
    "value_used_for_alignment":
        False,
    "unit_used_for_alignment":
        False,
    "category_used_for_alignment":
        False,
    "topic_used_for_alignment":
        False,
    "description_used_for_alignment":
        False,
    "source_location_used_for_alignment":
        False
}

content_diagnostics = {
    "reference_record_count_valid":
        reference_record_count_valid,
    "reference_category_counts_valid":
        reference_category_counts_valid,
    "extraction_record_count_valid":
        extracted_count == EXPECTED_RECORD_COUNT,
    "extraction_category_counts_valid":
        dict(
            Counter(
                record.get("Category")
                for record in extracted_records
                if isinstance(record, dict)
            )
        ) == EXPECTED_CATEGORY_COUNTS,
    "reference_identity_unique":
        reference_identity_unique,
    "extraction_duplicate_identity_count":
        len(extraction_duplicates),
    "ambiguous_identity_group_count":
        len(alignment_issues)
}

schema_diagnostics = {
    "top_level_object_valid":
        top_level_object_valid,
    "document_id_correct":
        document_id_correct,
    "branch_correct":
        branch_correct,
    "records_is_list":
        records_is_list,
    "record_schema_valid":
        record_schema_valid,
    "field_types_valid":
        field_types_valid,
    "schema_validity":
        schema_validity
}

VALIDATION_SUMMARY = {
    "document_id":
        DOCUMENT_ID,
    "document_name":
        DOCUMENT_NAME,
    "branch":
        BRANCH,
    "branch_name":
        BRANCH_NAME,
    "input_representation":
        INPUT_REPRESENTATION,
    "reference_records":
        reference_count,
    "extracted_records":
        extracted_count,
    "aligned_records":
        aligned_count,
    "fully_correct_records":
        fully_correct_count,
    "discrepant_records":
        discrepant_count,
    "missing_records":
        missing_count,
    "unsupported_extracted_records":
        unsupported_count,
    "completeness":
        completeness,
    "missing_rate":
        missing_count / reference_count
        if reference_count else None,
    "record_precision_exact":
        record_precision_exact,
    "record_recall_exact":
        record_recall_exact,
    "record_f1_exact":
        record_f1_exact,
    "unsupported_rate":
        unsupported_count / extracted_count
        if extracted_count else None,
    "discrepancy_rate_among_aligned":
        discrepant_count / aligned_count
        if aligned_count else None,
    "field_accuracy":
        field_accuracy,
    "primary_field_accuracy":
        primary_field_accuracy,
    "schema_validity":
        schema_validity,
    "schema_diagnostics":
        schema_diagnostics,
    "content_diagnostics":
        content_diagnostics,
    "matching_rules":
        matching_rules,
    "comparison_rules":
        comparison_rules,
    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            reference_semantics_valid,
        "checks":
            reference_semantic_checks,
        "reference_modified_by_validation":
            False
    },
    "category_metrics":
        category_metrics,
    "input_provenance":
        input_provenance,
}

print(json.dumps(
    VALIDATION_SUMMARY,
    ensure_ascii=False,
    indent=2
))

assert (
    aligned_count
    + missing_count
    == reference_count
)

assert (
    aligned_count
    + unsupported_count
    == extracted_count
)

assert (
    fully_correct_count
    + discrepant_count
    == aligned_count
)

assert schema_validity == bool(
    branch_A_structurally_evaluable
)

In [ ]:
# ============================================================
# 12. Export validation outputs
# ============================================================

detailed_df.to_csv(
    PATHS["detailed"],
    index=False,
    encoding="utf-8-sig"
)

discrepant_df.to_csv(
    PATHS["discrepant"],
    index=False,
    encoding="utf-8-sig"
)

missing_df.to_csv(
    PATHS["missing"],
    index=False,
    encoding="utf-8-sig"
)

unsupported_df.to_csv(
    PATHS["unsupported"],
    index=False,
    encoding="utf-8-sig"
)

PATHS["summary"].write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

missing_outputs = [
    path.name
    for path in PATHS.values()
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing validation outputs: {missing_outputs}"
    )

print("Validation A — D12 completed successfully.")
print("Fully correct records:", fully_correct_count)
print("Discrepant records:", discrepant_count)
print("Missing records:", missing_count)
print("Unsupported/unmatched records:", unsupported_count)
print("Exact F1:", record_f1_exact)
print("Schema validity:", schema_validity)
print(
    "Branch A structurally evaluable:",
    branch_A_structurally_evaluable
)

print(
    "Field accuracy:",
    field_accuracy
)

print("\nGenerated files:")
for path in PATHS.values():
    print("-", path.name)